Step 6- This EDA is needed to check if the SYMPTOM1-5 are all ADEs. It can contain DRUG names and other administrative cases that shouldn't be marked as ADE

In [ ]:
import os
import re
import pandas as pd
from collections import Counter
from datetime import datetime

# ---------- Paths ----------
SRC = os.path.join("..", "data", "processed", "sample_1k_with_severity.csv")

# ---------- Load ----------
df = pd.read_csv(SRC, low_memory=False)

# ---------- Helpers ----------
def norm(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def top_counts(series, n=30):
    s = (series.dropna().astype(str).str.strip())
    s = s[s.ne("")]  # remove empty
    return s.value_counts().head(n)

def parse_date(mdY):
    try:
        return datetime.strptime(str(mdY), "%m/%d/%Y")
    except Exception:
        return None

# Post-vaccine COVID verifier (conservative)
POS_PATTERNS = [
    r"tested\s+positive", r"pcr\s+positive", r"antigen\s+positive",
    r"diagnosed\s+with\s+covid", r"developed\s+covid",
    r"symptomatic\s+covid", r"breakthrough\s+infection",
    r"(?:\b\d{1,3}\b)\s*(?:day|week|month)s?\s*(?:after|post)\s*(?:vaccine|vaccination|shot|dose)",
    r"post[-\s]*vaccination", r"following\s+vaccination"
]
NEG_PATTERNS = [
    r"prior\s+to\s+vaccination", r"before\s+vaccination", r"history\s+of\s+covid",
    r"previous\s+covid", r"vaccinated\s+after\s+testing\s+positive", r"pre[-\s]*existing\s+covid"
]
POS_RE = re.compile("|".join(POS_PATTERNS), re.IGNORECASE)
NEG_RE = re.compile("|".join(NEG_PATTERNS), re.IGNORECASE)
COVID_RE = re.compile(r"\b(covid[-\s]*19|sars[-\s]*cov[-\s]*2|covid)\b", re.IGNORECASE)
TEST_POS_RE = re.compile(r"(?:pcr|antigen|rt[-\s]*pcr|test)\s+positive", re.IGNORECASE)

def looks_post_vax_covid(row) -> bool:
    text = str(row.get("SYMPTOM_TEXT", "") or "")
    if not COVID_RE.search(text):
        return False
    vax = parse_date(row.get("VAX_DATE"))
    onset = parse_date(row.get("ONSET_DATE"))
    if vax and onset and onset >= vax:
        return True
    try:
        numdays = float(row.get("NUMDAYS"))
        if numdays >= 0:
            return True
    except Exception:
        pass
    if NEG_RE.search(text):
        return False
    if POS_RE.search(text) or TEST_POS_RE.search(text):
        return True
    return False

# ---------- Columns present ----------
sym_cols = [c for c in ["SYMPTOM1","SYMPTOM2","SYMPTOM3","SYMPTOM4","SYMPTOM5"] if c in df.columns]
print(f"Symptom columns found: {sym_cols}")

# ---------- Top 20 per SYMPTOM1–3 ----------
for c in ["SYMPTOM1","SYMPTOM2","SYMPTOM3"]:
    if c in df.columns:
        print(f"\n=== Top 20 {c} ===")
        print(top_counts(df[c], 20))

# ---------- Combined top symptoms across SYMPTOM1–5 ----------
combined = []
for c in sym_cols:
    combined.extend(df[c].dropna().astype(str).tolist())
combined_norm = [norm(x) for x in combined if str(x).strip()]
ctr = Counter(combined_norm)
print("\n=== Combined top 30 across SYMPTOM1–5 (normalized) ===")
for k,v in ctr.most_common(30):
    print(f"{k}\t{v}")

# ---------- Vaccine names (for leakage & DRUG lexicon) ----------
if "VAX_NAME" in df.columns:
    vax_names = sorted({norm(x) for x in df["VAX_NAME"].dropna().astype(str)})
    print(f"\nUnique vaccine names: {len(vax_names)}")

    sym_norm_all = set(combined_norm)
    leaked = sym_norm_all.intersection(set(vax_names))
    if leaked:
        print("\nVaccine names appearing inside SYMPTOM fields (should not be ADE):")
        for name in sorted(list(leaked))[:30]:
            print(" -", name)
    else:
        print("\nNo vaccine-name leakage detected in SYMPTOM fields.")

# ---------- COVID / SARS-CoV-2 mention counts ----------
text_series = df.get("SYMPTOM_TEXT")
if text_series is not None:
    text_nonnull = text_series.dropna().astype(str)
    total_text = len(text_nonnull)
    covid_mentions = text_nonnull.str.contains(COVID_RE).sum()
    test_pos_mentions = text_nonnull.str.contains(TEST_POS_RE).sum()
    print(f"\nSYMPTOM_TEXT rows (non-null): {total_text}")
    print(f"Mentions of COVID/SARS-CoV-2 (any): {covid_mentions}")
    print(f"Mentions of 'test positive' patterns: {test_pos_mentions}")
    post_vax_flags = df.apply(looks_post_vax_covid, axis=1)
    print(f"Likely post-vaccine COVID cases (rule-based): {post_vax_flags.sum()}")

# ---------- Blacklist ----------
BLACKLIST = {
    # admin/product/handling errors
    "expired product administered",
    "product storage error",
    "incorrect dose administered",
    "product administered to patient of inappropriate age",
    "inappropriate schedule of product administration",
    "vaccination error",
    "accidental exposure",

    # negations / non-events
    "no adverse event",
    "normal test result",
    "test negative",
    "unevaluable event",   # <--- NEWLY ADDED

    # diagnostics/procedures
    "blood test", "laboratory test", "x-ray", "ct scan",

    # ambiguous disease label (handle via narrative rule, not lexicon)
    "covid-19", "covid 19", "covid19", "sars-cov-2", "sars cov 2"
}

blacklist_counts = {term: ctr.get(term, 0) for term in BLACKLIST}
print("\n=== Blacklist terms frequency (in combined SYMPTOM1–5) ===")
for term, count in sorted(blacklist_counts.items(), key=lambda x: -x[1]):
    if count > 0:
        print(f"{term}\t{count}")

removed_total = sum(blacklist_counts.values())
print(f"\nEstimated SYMPTOM entries removed by blacklist: {removed_total}")

if "VAX_NAME" in df.columns:
    vaccine_leak_count = sum(ctr.get(name, 0) for name in vax_names)
    print(f"SYMPTOM entries that are exact vaccine names (to exclude from ADE): {vaccine_leak_count}")

print("\n✅ EDA complete.")


Symptom columns found: ['SYMPTOM1', 'SYMPTOM2', 'SYMPTOM3', 'SYMPTOM4', 'SYMPTOM5']

=== Top 20 SYMPTOM1 ===
SYMPTOM1
Chills                          70
COVID-19                        67
Arthralgia                      53
Expired product administered    32
Dizziness                       31
Fatigue                         27
Asthenia                        22
Erythema                        21
Headache                        20
Injection site erythema         16
Product storage error           14
Rash                            13
Blood test                      12
Pain in extremity               12
Condition aggravated            12
Hypoaesthesia                   11
Pruritus                        11
Injection site pain             10
Chest discomfort                 9
Unevaluable event                9
Name: count, dtype: int64

=== Top 20 SYMPTOM2 ===
SYMPTOM2
Headache                    41
Fatigue                     40
Chills                      32
Dizziness                   2

C:\Users\Mochitha vijayan\AppData\Local\Temp\ipykernel_20920\712059299.py:106: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  covid_mentions = text_nonnull.str.contains(COVID_RE).sum()



SYMPTOM_TEXT rows (non-null): 1000
Mentions of COVID/SARS-CoV-2 (any): 386
Mentions of 'test positive' patterns: 4
Likely post-vaccine COVID cases (rule-based): 269

=== Blacklist terms frequency (in combined SYMPTOM1–5) ===
covid-19	83
expired product administered	33
product storage error	21
no adverse event	21
blood test	19
inappropriate schedule of product administration	11
laboratory test	9
unevaluable event	9
product administered to patient of inappropriate age	6
incorrect dose administered	6
vaccination error	1
x-ray	1

Estimated SYMPTOM entries removed by blacklist: 220
SYMPTOM entries that are exact vaccine names (to exclude from ADE): 0

✅ EDA complete.
